# Binance Crypto — Quantitative EDA

Exploratory Data Analysis del pipeline de datos históricos de Binance.

**Datos:** BTCUSDT 1h, Junio 2021 – Mayo 2026 (43,817 barras)

**Objetivo:** Entender distribución de retornos, patrones de estacionalidad y fundamentos estadísticos de las hipótesis H2 y H4.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

# Configuración de gráficas
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded.')

In [ ]:
from src.ingestion import load_symbol
from src.transform import add_returns, add_features

df = load_symbol('BTCUSDT')
df = add_returns(df)
df = add_features(df)

print(f'Shape: {df.shape}')
print(f'Rango: {df["open_time_utc"].min().date()} → {df["open_time_utc"].max().date()}')
df.head(3)

## 1. Precio de cierre — serie de tiempo

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Precio
axes[0].plot(df['open_time_utc'], df['close'], linewidth=0.8, color='#F7931A')
axes[0].set_title('BTCUSDT — Precio de cierre (1h)', fontsize=13)
axes[0].set_ylabel('USD')

# Retorno diario acumulado
cum_return = (1 + df['open_to_open_return'].fillna(0)).cumprod() - 1
axes[1].plot(df['open_time_utc'], cum_return * 100, linewidth=0.8, color='#2196F3')
axes[1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[1].set_title('Retorno acumulado open-to-open (%)', fontsize=13)
axes[1].set_ylabel('%')

plt.tight_layout()
plt.show()

## 2. Distribución de retornos por día de la semana

In [ ]:
weekday_names = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

df_wd = (
    df.dropna(subset=['open_to_open_return', 'next_bar_weekday'])
    .groupby('next_bar_weekday')['open_to_open_return']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
df_wd['weekday_name'] = df_wd['next_bar_weekday'].map(dict(enumerate(weekday_names)))
df_wd['mean_pct'] = df_wd['mean'] * 100

colors = ['#2ECC71' if v >= 0 else '#E74C3C' for v in df_wd['mean_pct']]
plt.bar(df_wd['weekday_name'], df_wd['mean_pct'], color=colors, edgecolor='white')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Retorno promedio open-to-open por día de la semana (BTCUSDT 1h)', fontsize=13)
plt.ylabel('Retorno promedio (%)')
plt.xlabel('Día de la semana (de la barra activa)')
plt.tight_layout()
plt.show()

print(df_wd[['weekday_name','mean_pct','std','count']].to_string(index=False))

## 3. Volatilidad rolling — 30 días

In [ ]:
vol_daily = df.set_index('open_time_utc')['open_to_open_return'].resample('D').std() * np.sqrt(365)
vol_30d = vol_daily.rolling(30).mean()

plt.figure(figsize=(14, 4))
plt.plot(vol_30d.index, vol_30d * 100, color='#9B59B6', linewidth=1)
plt.fill_between(vol_30d.index, vol_30d * 100, alpha=0.2, color='#9B59B6')
plt.title('Volatilidad anualizada rolling 30 días (%)', fontsize=13)
plt.ylabel('Volatilidad (%)')
plt.tight_layout()
plt.show()

## 4. Régimen de mercado — distribución de barras

In [ ]:
regime_counts = df['market_regime'].value_counts()

plt.figure(figsize=(6, 5))
colors = {'Bull': '#2ECC71', 'Bear': '#E74C3C', 'Lateral': '#F39C12'}
plt.pie(
    regime_counts.values,
    labels=regime_counts.index,
    colors=[colors.get(r, 'gray') for r in regime_counts.index],
    autopct='%1.1f%%',
    startangle=90
)
plt.title('Distribución de barras por régimen de mercado', fontsize=13)
plt.tight_layout()
plt.show()
print(regime_counts)